<a href="https://colab.research.google.com/github/Prasadn1234/DataScience/blob/main/Copy_of_Travel_Planning_Flight_Hotel_Booking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accessible Travel Planning Team

Converted from `n8n Workflow - Accessible Travel Planning Team.json`.

This notebook is the same five-agent pipeline, rewritten so it runs in Python. Google Sheets tools read the local CSVs. The Lisbon weather tool calls the same Open-Meteo URL. Each agent still returns the fixed fields from the n8n system prompts.

**Chain:** chat message → Coordinator → Flight Agent → Itinerary Agent → Constraint Checker.

**Manager and worker:** the Itinerary Agent calls the Hotel Agent. The Hotel Agent is not on the main line.

**Quality gate:** the Constraint Checker returns `APPROVED` or `NEEDS REVISION`.

Supported route only: **San Francisco (SFO) → Lisbon (LIS)**.


## Setup

The n8n chat trigger is the `traveler_message` string passed into `plan_trip`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install anthropic pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.3 MB/s  0:00:00


In [ ]:
import os, json, io
import pandas as pd
import requests
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-haiku-5-5"   # fast/cheap analog of gpt-4o-mini; try "claude-sonnet-5-5" for higher accuracy
MAX_ITERATIONS = 10          # same as the n8n agent nodes
VERBOSE = True               # print every tool call (like n8n's Logs panel)

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import requests

BASE = Path(r"/content/drive/MyDrive/Travel_Hotel_Booking/Datasets-Flight_Hotel/Datasets")
FLIGHTS_CSV = BASE / "flights_dataset.csv"
HOTELS_CSV = BASE / "hotels_dataset.csv"

WEATHER_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=38.72&longitude=-9.14"
    "&start_date=2025-10-10&end_date=2025-10-15"
    "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum"
    "&timezone=Europe/Lisbon"
)

UNSUPPORTED = (
    "UNSUPPORTED_DESTINATION: This system only supports San Francisco to Lisbon."
)

MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}
display(FLIGHTS_CSV)
flights = pd.read_csv(FLIGHTS_CSV, dtype=str)
display(flights)

PosixPath('/content/drive/MyDrive/Travel_Hotel_Booking/Datasets-Flight_Hotel/Datasets/flights_dataset.csv')

,flight_id,airline,origin,destination,departure,arrival,stops,price_usd,wheelchair_assistance,extra_legroom_seats
0,FL101,SkyWay Air,SFO,LIS,2026-10-12 08:15,2026-10-12 21:40,0,410,yes,yes
1,FL102,SkyWay Air,SFO,LIS,2026-10-12 14:20,2026-10-13 09:05,1,295,yes,no
2,FL103,AtlanticBlue,SFO,LIS,2026-10-12 07:00,2026-10-12 20:10,0,545,yes,yes
3,FL104,ValueJet,SFO,LIS,2026-10-12 22:10,2026-10-13 18:45,2,210,no,no
4,FL105,AtlanticBlue,SFO,LIS,2026-10-13 09:30,2026-10-13 22:55,0,470,yes,yes
5,FL106,SkyWay Air,SFO,LIS,2026-10-13 11:00,2026-10-14 03:15,1,260,yes,no


## Datasets

Stand-ins for the **Get Flights** and **Get Hotels** Google Sheets tools. Departure and arrival stay as plain text (`2026-10-12 08:15`).

In [ ]:
flights = pd.read_csv(FLIGHTS_CSV, dtype=str)
hotels = pd.read_csv(HOTELS_CSV, dtype=str)

for col in ("stops", "price_usd"):
    flights[col] = pd.to_numeric(flights[col])
for col in ("price_per_night_usd", "distance_to_center_km"):
    hotels[col] = pd.to_numeric(hotels[col])

flights["departure_date"] = flights["departure"].str.slice(0, 10)

display(flights)
display(hotels)

,flight_id,airline,origin,destination,departure,arrival,stops,price_usd,wheelchair_assistance,extra_legroom_seats,departure_date
0,FL101,SkyWay Air,SFO,LIS,2026-10-12 08:15,2026-10-12 21:40,0,410,yes,yes,2026-10-12
1,FL102,SkyWay Air,SFO,LIS,2026-10-12 14:20,2026-10-13 09:05,1,295,yes,no,2026-10-12
2,FL103,AtlanticBlue,SFO,LIS,2026-10-12 07:00,2026-10-12 20:10,0,545,yes,yes,2026-10-12
3,FL104,ValueJet,SFO,LIS,2026-10-12 22:10,2026-10-13 18:45,2,210,no,no,2026-10-12
4,FL105,AtlanticBlue,SFO,LIS,2026-10-13 09:30,2026-10-13 22:55,0,470,yes,yes,2026-10-13
5,FL106,SkyWay Air,SFO,LIS,2026-10-13 11:00,2026-10-14 03:15,1,260,yes,no,2026-10-13


,hotel_id,name,neighborhood,price_per_night_usd,wheelchair_accessible_room,step_free_entrance,accessible_bathroom,distance_to_center_km
0,HT201,Hotel Belem Central,Belem,140,yes,yes,yes,3.2
1,HT202,Lisbon Harbor Inn,Alfama,95,no,yes,no,1.1
2,HT203,Praca Suites,Baixa,180,yes,yes,yes,0.4
3,HT204,Riverside Budget Stay,Alcantara,70,no,no,no,4.8
4,HT205,Alfama Accessible Rooms,Alfama,155,yes,yes,yes,1.3
5,HT206,Baixa Boutique Hotel,Baixa,210,yes,yes,no,0.2


## Agent 1 — Coordinator

Reads the traveler message and writes a fixed brief. It does not search flights or hotels. Any destination other than Lisbon stops the pipeline with `UNSUPPORTED_DESTINATION`. A missing year is treated as 2026. Missing fields stay `not specified`.


In [ ]:
@dataclass
class TripBrief:
    destination: str
    travel_date: str
    budget_total_usd: str
    direct_flight_required: str
    accessibility_needs: str
    nights: str
    notes: str
    unsupported: bool = False
    raw: str = ""

    def render(self) -> str:
        if self.unsupported:
            return UNSUPPORTED
        return (
            f"DESTINATION: {self.destination}\n"
            f"TRAVEL_DATE: {self.travel_date}\n"
            f"BUDGET_TOTAL_USD: {self.budget_total_usd}\n"
            f"DIRECT_FLIGHT_REQUIRED: {self.direct_flight_required}\n"
            f"ACCESSIBILITY_NEEDS: {self.accessibility_needs}\n"
            f"NIGHTS: {self.nights}\n"
            f"NOTES: {self.notes}"
        )


def _travel_date(text: str) -> str:
    iso = re.search(r"\b(\d{4})-(\d{2})-(\d{2})\b", text)
    if iso:
        return iso.group(0)
    named = re.search(
        r"\b([A-Za-z]{3,9})\s+(\d{1,2})(?:st|nd|rd|th)?(?:,)?\s*(\d{4})?",
        text,
        flags=re.I,
    )
    if not named:
        return "not specified"
    month = MONTHS.get(named.group(1).lower())
    if not month:
        return "not specified"
    year = named.group(3) or "2026"
    return f"{int(year):04d}-{month:02d}-{int(named.group(2)):02d}"


def _destination(text: str) -> tuple[str, bool]:
    match = re.search(
        r"\bto\s+([A-Za-z][A-Za-z .'-]{1,40}?)(?=\s+on\b|\s+for\b|\s+from\b|[.,]|$)",
        text,
        flags=re.I,
    )
    if not match:
        if re.search(r"\b(lisbon|lisboa|\blis\b)\b", text, flags=re.I):
            return "LIS", False
        return "not specified", False
    city = re.sub(r"\s+", " ", match.group(1)).strip(" .")
    if city.lower() in {"lisbon", "lisboa", "lis"}:
        return "LIS", False
    return city, True


def _needs(text: str) -> str:
    low = text.lower()
    found = []
    if "wheelchair assistance" in low or "wheelchair assist" in low:
        found.append("wheelchair assistance")
    if "wheelchair-accessible room" in low or "wheelchair accessible room" in low:
        found.append("wheelchair-accessible room")
    if "step-free" in low or "step free" in low:
        found.append("step-free entrance")
    if "accessible bathroom" in low:
        found.append("accessible bathroom")
    return ", ".join(found) if found else "not specified"


def coordinator(message: str) -> TripBrief:
    """Agent 1: traveler request -> structured brief."""
    destination, unsupported = _destination(message)
    budget = re.search(r"\$\s*(\d+(?:\.\d+)?)", message)
    nights = re.search(r"\b(\d+)\s*nights?\b", message, flags=re.I)
    direct = "yes" if re.search(r"\bdirect\b", message, flags=re.I) else "no"
    brief = TripBrief(
        destination=destination,
        travel_date=_travel_date(message),
        budget_total_usd=budget.group(1) if budget else "not specified",
        direct_flight_required=direct if not unsupported else "not specified",
        accessibility_needs=_needs(message),
        nights=nights.group(1) if nights else "not specified",
        notes="none",
        unsupported=unsupported,
    )
    if unsupported:
        brief.direct_flight_required = "not specified"
    brief.raw = brief.render()
    return brief

## Agent 2 — Flight Agent

Tool: **Get Flights** (every row in `flights_dataset.csv`).

Filters, in order:

1. Departure date equals `TRAVEL_DATE`.
2. If a direct flight is required, `stops = 0`.
3. If wheelchair assistance is required, `wheelchair_assistance = yes`.
4. `price_usd` is less than or equal to `BUDGET_TOTAL_USD`.
5. Pick the cheapest remaining flight. The alternate is the next cheapest.

`UNSUPPORTED_DESTINATION` is passed through and the tool is not used. If nothing survives the budget filter, the reply starts with `NO_MATCHING_FLIGHT` and names the cheapest flight that passed steps 1–3.


In [ ]:
@dataclass
class FlightPick:
    status: str
    text: str
    flight: pd.Series | None = None
    alternate_id: str = "none available"
    ignored_budget: pd.Series | None = None


def get_flights() -> pd.DataFrame:
    """Get Flights tool: full flights dataset, no arguments."""
    return flights.copy()


def flight_agent(brief: TripBrief) -> FlightPick:
    """Agent 2: brief -> one recommended flight."""
    if brief.unsupported:
        return FlightPick("unsupported", UNSUPPORTED)

    data = get_flights()
    data = data[data["departure_date"] == brief.travel_date]
    if brief.direct_flight_required == "yes":
        data = data[data["stops"] == 0]
    needs = brief.accessibility_needs.lower()
    if "wheelchair assistance" in needs:
        data = data[data["wheelchair_assistance"].str.lower() == "yes"]

    pool = data.sort_values(["price_usd", "flight_id"])
    if pool.empty or brief.budget_total_usd == "not specified":
        return FlightPick(
            "no_match",
            "NO_MATCHING_FLIGHT: no flight remains after the date, direct, and accessibility filters.",
        )

    budget = float(brief.budget_total_usd)
    affordable = pool[pool["price_usd"] <= budget]
    if affordable.empty:
        cheapest = pool.iloc[0]
        text = (
            "NO_MATCHING_FLIGHT: no flight on this date passes the direct-flight and "
            f"accessibility filters at or under ${budget:.0f}. "
            f"Cheapest flight ignoring budget: {cheapest.flight_id} ${int(cheapest.price_usd)}."
        )
        return FlightPick("no_match", text, ignored_budget=cheapest)

    chosen = affordable.iloc[0]
    alternate = affordable.iloc[1].flight_id if len(affordable) > 1 else "none available"
    access = (
        "yes, wheelchair_assistance is yes"
        if "wheelchair assistance" in needs
        else "yes, no wheelchair-assistance constraint was set"
    )
    text = (
        f"RECOMMENDED_FLIGHT_ID: {chosen.flight_id}\n"
        f"AIRLINE: {chosen.airline}\n"
        f"DEPARTURE: {chosen.departure}\n"
        f"PRICE_USD: {int(chosen.price_usd)}\n"
        f"STOPS: {int(chosen.stops)}\n"
        f"ACCESSIBILITY_MATCH: {access}\n"
        f"ALTERNATE_OPTION: {alternate}"
    )
    return FlightPick("ok", text, flight=chosen, alternate_id=alternate)

## Agent 4 — Hotel Agent (worker tool)

Called only by the Itinerary Agent. Tool: **Get Hotels**.

1. Wheelchair-accessible room, step-free access, or wheelchair access keeps hotels with both `wheelchair_accessible_room = yes` and `step_free_entrance = yes`.
2. An accessible-bathroom need also requires `accessible_bathroom = yes`.
3. `ESTIMATED_TOTAL_USD = price_per_night_usd × nights`.
4. Pick the cheapest nightly rate. A price tie goes to the hotel closer to the center.
5. The alternate is the next cheapest hotel.

The hotel agent does not apply the trip budget. The Constraint Checker does that later.


In [ ]:
@dataclass
class HotelPick:
    status: str
    text: str
    hotel: pd.Series | None = None
    nights: int = 0
    estimated_total: float = 0
    alternate_id: str = "none available"


def get_hotels() -> pd.DataFrame:
    """Get Hotels tool: full hotels dataset, no arguments."""
    return hotels.copy()


def hotel_agent(brief: TripBrief) -> HotelPick:
    """Agent 4: nights, budget, and accessibility needs -> one hotel."""
    needs = brief.accessibility_needs.lower()
    nights = int(brief.nights) if brief.nights != "not specified" else 0
    data = get_hotels()

    room_or_step = any(
        token in needs
        for token in (
            "wheelchair-accessible room",
            "wheelchair accessible room",
            "step-free",
            "step free",
            "wheelchair access",
        )
    )
    if room_or_step:
        data = data[
            (data["wheelchair_accessible_room"].str.lower() == "yes")
            & (data["step_free_entrance"].str.lower() == "yes")
        ]
    if "accessible bathroom" in needs:
        data = data[data["accessible_bathroom"].str.lower() == "yes"]

    if data.empty or nights <= 0:
        reason = (
            "no hotel meets the accessibility constraints"
            if data.empty
            else "nights were not specified"
        )
        return HotelPick("no_match", f"NO_MATCHING_HOTEL: {reason}.")

    ranked = data.sort_values(["price_per_night_usd", "distance_to_center_km", "hotel_id"])
    chosen = ranked.iloc[0]
    alternate = ranked.iloc[1].hotel_id if len(ranked) > 1 else "none available"
    total = float(chosen.price_per_night_usd) * nights
    why = "yes, required accessibility flags match the dataset row"
    text = (
        f"RECOMMENDED_HOTEL_ID: {chosen.hotel_id}\n"
        f"NAME: {chosen.name}\n"
        f"PRICE_PER_NIGHT_USD: {int(chosen.price_per_night_usd)}\n"
        f"NIGHTS: {nights}\n"
        f"ESTIMATED_TOTAL_USD: {int(total)} ({int(chosen.price_per_night_usd)} x {nights})\n"
        f"ACCESSIBILITY_MATCH: {why}\n"
        f"ALTERNATE_OPTION: {alternate}"
    )
    return HotelPick("ok", text, hotel=chosen, nights=nights, estimated_total=total, alternate_id=alternate)

## Agent 3 — Itinerary Agent (manager)

Joins the Coordinator brief and the Flight Agent result. It calls the Hotel Agent once, passing nights, total budget, and every accessibility need.

It does not call the Hotel Agent when the input contains `UNSUPPORTED_DESTINATION` or `NO_MATCHING_FLIGHT`. A `NO_MATCHING_HOTEL` result still produces an itinerary, with `MISSING` on the hotel line.


In [ ]:
@dataclass
class Itinerary:
    status: str
    text: str
    flight: FlightPick
    hotel: HotelPick | None = None
    total: float | None = None


def itinerary_agent(brief: TripBrief, flight: FlightPick) -> Itinerary:
    """Agent 3: brief + flight -> draft plan. Calls Hotel Agent as a tool."""
    if brief.unsupported or flight.status == "unsupported":
        return Itinerary("unsupported", UNSUPPORTED, flight)
    if flight.status == "no_match":
        return Itinerary("no_flight", flight.text, flight)

    hotel = hotel_agent(brief)
    if hotel.status != "ok" or flight.flight is None:
        flight_price = int(flight.flight.price_usd) if flight.flight is not None else "MISSING"
        flight_line = (
            f"{flight.flight.flight_id}, {flight.flight.airline}, ${int(flight.flight.price_usd)}, {int(flight.flight.stops)} stops"
            if flight.flight is not None
            else "MISSING"
        )
        text = (
            f"SUMMARY: {brief.destination} on {brief.travel_date}\n"
            f"FLIGHT: {flight_line}\n"
            "HOTEL: MISSING\n"
            f"TOTAL_ESTIMATED_COST_USD: {flight_price}\n"
            "TRAVELER_CONSTRAINTS: "
            f"budget ${brief.budget_total_usd}, direct flight {brief.direct_flight_required}, "
            f"accessibility {brief.accessibility_needs}\n"
            f"OPEN_ITEMS: {hotel.text}"
        )
        return Itinerary("no_hotel", text, flight, hotel)

    flight_price = float(flight.flight.price_usd)
    total = flight_price + hotel.estimated_total
    text = (
        f"SUMMARY: {brief.destination} on {brief.travel_date}, {hotel.nights} nights\n"
        f"FLIGHT: {flight.flight.flight_id}, {flight.flight.airline}, ${int(flight_price)}, {int(flight.flight.stops)} stops\n"
        f"HOTEL: {hotel.hotel.hotel_id}, {hotel.hotel.name}, "
        f"${int(hotel.hotel.price_per_night_usd)}/night x {hotel.nights} = ${int(hotel.estimated_total)}\n"
        f"TOTAL_ESTIMATED_COST_USD: {int(total)}\n"
        "TRAVELER_CONSTRAINTS: "
        f"budget ${brief.budget_total_usd}, direct flight {brief.direct_flight_required}, "
        f"accessibility {brief.accessibility_needs}\n"
        "OPEN_ITEMS: none"
    )
    return Itinerary("ok", text, flight, hotel, total)

## Agent 5 — Constraint Checker

Final quality gate. It does not invent a new flight or hotel. Tool: **Lisbon Weather** (fixed Open-Meteo URL, no API key).

Checks:

1. **Budget** — flight price + hotel total ≤ `BUDGET_TOTAL_USD`.
2. **Direct flight** — 0 stops when required, otherwise `N/A`.
3. **Accessibility** — the chosen flight and hotel satisfy every need in the brief.
4. **Weather** — risk if any day has rainfall above 10 mm, a maximum above 35 °C, or a minimum below 2 °C.

`APPROVED` only when every check is `PASS` or `N/A`.


In [ ]:
def lisbon_weather() -> dict:
    """Lisbon Weather tool: fixed URL, no arguments."""
    response = requests.get(WEATHER_URL, timeout=30)
    response.raise_for_status()
    return response.json()


def _weather_check(payload: dict) -> tuple[str, bool]:
    daily = payload["daily"]
    risks = []
    for day, tmax, tmin, rain in zip(
        daily["time"],
        daily["temperature_2m_max"],
        daily["temperature_2m_min"],
        daily["precipitation_sum"],
    ):
        flags = []
        if rain is not None and rain > 10:
            flags.append(f"rainfall {rain} mm")
        if tmax is not None and tmax > 35:
            flags.append(f"max {tmax} C")
        if tmin is not None and tmin < 2:
            flags.append(f"min {tmin} C")
        if flags:
            risks.append(f"{day}: " + ", ".join(flags))
    if risks:
        return "RISK — " + "; ".join(risks), False
    return "PASS — conditions look fine (no day over the rain or temperature limits)", True


def _accessibility_ok(brief: TripBrief, itinerary: Itinerary) -> tuple[str, bool]:
    needs = brief.accessibility_needs.lower()
    flight = itinerary.flight.flight
    hotel = itinerary.hotel.hotel if itinerary.hotel else None
    problems = []
    if "wheelchair assistance" in needs:
        if flight is None or str(flight.wheelchair_assistance).lower() != "yes":
            problems.append("flight is missing wheelchair assistance")
    if hotel is None:
        problems.append("hotel is missing")
    else:
        if any(token in needs for token in ("wheelchair-accessible room", "wheelchair accessible room", "step-free", "step free", "wheelchair access")):
            if str(hotel.wheelchair_accessible_room).lower() != "yes" or str(hotel.step_free_entrance).lower() != "yes":
                problems.append("hotel is missing a wheelchair-accessible room or step-free entrance")
        if "accessible bathroom" in needs and str(hotel.accessible_bathroom).lower() != "yes":
            problems.append("hotel is missing an accessible bathroom")
    if problems:
        return "FAIL, " + "; ".join(problems), False
    return "PASS, chosen flight and hotel match every accessibility need in the brief", True


def constraint_checker(brief: TripBrief, itinerary: Itinerary) -> str:
    """Agent 5: brief + draft itinerary -> verdict the traveler sees."""
    if brief.unsupported or itinerary.status == "unsupported":
        return "No data available for this destination. This system supports San Francisco to Lisbon only."

    if itinerary.status == "no_flight":
        return (
            "BUDGET_CHECK: FAIL, no flight available within the stated budget\n"
            f"{itinerary.flight.text}\n"
            "OVERALL_VERDICT: NEEDS REVISION"
        )

    flight = itinerary.flight.flight
    hotel_pick = itinerary.hotel
    hotel = hotel_pick.hotel if hotel_pick else None
    if flight is None or hotel is None or itinerary.total is None:
        return (
            "ITINERARY_UNDER_REVIEW:\n"
            "FLIGHT: MISSING\n"
            "HOTEL: MISSING\n"
            "TOTAL: MISSING\n"
            "BUDGET_CHECK: FAIL, itinerary is incomplete\n"
            "DIRECT_FLIGHT_CHECK: FAIL\n"
            "ACCESSIBILITY_CHECK: FAIL, flight or hotel is missing\n"
            "WEATHER_CHECK: not run\n"
            "OVERALL_VERDICT: NEEDS REVISION\n"
            "IF_NEEDS_REVISION: Hotel Agent or Flight Agent output is missing an id or a price"
        )

    flight_price = float(flight.price_usd)
    hotel_total = float(hotel_pick.estimated_total)
    total = flight_price + hotel_total
    budget = float(brief.budget_total_usd) if brief.budget_total_usd != "not specified" else None
    budget_ok = budget is not None and total <= budget
    budget_line = (
        f"{'PASS' if budget_ok else 'FAIL'}, ${int(flight_price)} + ${int(hotel_total)} = ${int(total)} "
        f"vs budget ${int(budget) if budget is not None else 'not specified'}"
    )

    if brief.direct_flight_required == "yes":
        direct_ok = int(flight.stops) == 0
        direct_line = "PASS, 0 stops" if direct_ok else f"FAIL, flight has {int(flight.stops)} stops"
    else:
        direct_ok = True
        direct_line = "N/A"

    access_line, access_ok = _accessibility_ok(brief, itinerary)
    weather_payload = lisbon_weather()
    weather_line, weather_ok = _weather_check(weather_payload)

    approved = budget_ok and direct_ok and access_ok and weather_ok
    if approved:
        revision = "not applicable"
        verdict = "APPROVED"
    else:
        failed = []
        if not budget_ok:
            failed.append("Flight Agent and Hotel Agent: total exceeds BUDGET_TOTAL_USD")
        if not direct_ok:
            failed.append("Flight Agent: chosen flight is not direct")
        if not access_ok:
            failed.append("Flight Agent or Hotel Agent: accessibility need not met")
        if not weather_ok:
            failed.append("weather risk on the sample Lisbon window")
        revision = "; ".join(failed)
        verdict = "NEEDS REVISION"

    return (
        "ITINERARY_UNDER_REVIEW:\n"
        f"FLIGHT: {flight.flight_id}, {flight.airline}, ${int(flight_price)}, {int(flight.stops)} stops\n"
        f"HOTEL: {hotel.hotel_id}, {hotel.name}, "
        f"${int(hotel.price_per_night_usd)}/night x {hotel_pick.nights} = ${int(hotel_total)}\n"
        f"TOTAL: ${int(total)}\n"
        f"BUDGET_CHECK: {budget_line}\n"
        f"DIRECT_FLIGHT_CHECK: {direct_line}\n"
        f"ACCESSIBILITY_CHECK: {access_line}\n"
        f"WEATHER_CHECK: {weather_line}\n"
        f"OVERALL_VERDICT: {verdict}\n"
        f"IF_NEEDS_REVISION: {revision}"
    )

## Pipeline

Same connections as the n8n workflow: Coordinator → Flight Agent → Itinerary Agent → Constraint Checker. Hotel Agent and Lisbon Weather run only when those agents call them.


In [ ]:
def plan_trip(traveler_message: str, show_steps: bool = True) -> str:
    """Chat trigger. Returns the Constraint Checker text."""
    brief = coordinator(traveler_message)
    flight = flight_agent(brief)
    draft = itinerary_agent(brief, flight)
    verdict = constraint_checker(brief, draft)
    if show_steps:
        print("=== Coordinator ===")
        print(brief.raw)
        print("\n=== Flight Agent ===")
        print(flight.text)
        print("\n=== Itinerary Agent ===")
        print(draft.text)
        print("\n=== Constraint Checker ===")
        print(verdict)
    return verdict

## Test prompts

Same five chats as the n8n sticky note. Each call is a new trip.

| # | Case | Expected |
| --- | --- | --- |
| 1 | Lisbon, 12 Oct 2026, 3 nights, $900, direct, wheelchair room and step-free entrance | FL101 $410 + HT201 $140×3 = $830, **APPROVED** |
| 2 | Same, but 13 Oct 2026 | FL105 $470 + HT201 $420 = $890, **APPROVED** |
| 3 | 12 Oct 2026, 4 nights, budget $800 | $410 + $560 = $970, **NEEDS REVISION** |
| 4 | 12 Oct 2026, 3 nights, budget $300 | **NO_MATCHING_FLIGHT** (FL101 $410); hotel tool not called |
| 5 | Paris | destination not supported |


In [ ]:
TESTS = [
    "Plan a trip from SFO to Lisbon on Oct 12, 2026. 3 nights, total budget $900. I need a direct flight, wheelchair assistance, and a hotel with a wheelchair-accessible room and step-free entrance.",
    "Plan a trip from SFO to Lisbon on Oct 13, 2026. 3 nights, total budget $900. I need a direct flight, wheelchair assistance, and a hotel with a wheelchair-accessible room and step-free entrance.",
    "Plan a trip from SFO to Lisbon on Oct 12, 2026. 4 nights, total budget $800. I need a direct flight, wheelchair assistance, and a hotel with a wheelchair-accessible room and step-free entrance.",
    "Plan a trip from SFO to Lisbon on Oct 12, 2026. 3 nights, total budget $300. I need a direct flight, wheelchair assistance, and a hotel with a wheelchair-accessible room and step-free entrance.",
    "Plan a trip from SFO to Paris on Oct 12, 2026, budget $1000.",
]

results = []
for index, prompt in enumerate(TESTS, start=1):
    print(f"\n########## Test {index} ##########")
    print(prompt)
    print()
    results.append(plan_trip(prompt))


########## Test 1 ##########
Plan a trip from SFO to Lisbon on Oct 12, 2026. 3 nights, total budget $900. I need a direct flight, wheelchair assistance, and a hotel with a wheelchair-accessible room and step-free entrance.

=== Coordinator ===
DESTINATION: LIS
TRAVEL_DATE: 2026-10-12
BUDGET_TOTAL_USD: 900
DIRECT_FLIGHT_REQUIRED: yes
ACCESSIBILITY_NEEDS: wheelchair assistance, wheelchair-accessible room, step-free entrance
NIGHTS: 3
NOTES: none

=== Flight Agent ===
RECOMMENDED_FLIGHT_ID: FL101
AIRLINE: SkyWay Air
DEPARTURE: 2026-10-12 08:15
PRICE_USD: 410
STOPS: 0
ACCESSIBILITY_MATCH: yes, wheelchair_assistance is yes
ALTERNATE_OPTION: FL103

=== Itinerary Agent ===
SUMMARY: LIS on 2026-10-12, 3 nights
FLIGHT: FL101, SkyWay Air, $410, 0 stops
HOTEL: HT201, 0, $140/night x 3 = $420
TOTAL_ESTIMATED_COST_USD: 830
TRAVELER_CONSTRAINTS: budget $900, direct flight yes, accessibility wheelchair assistance, wheelchair-accessible room, step-free entrance
OPEN_ITEMS: none

=== Constraint Checke

In [ ]:
assert "FL101" in results[0] and "HT201" in results[0] and "APPROVED" in results[0]
assert "$830" in results[0] or "830" in results[0]
assert "FL105" in results[1] and "APPROVED" in results[1]
assert "NEEDS REVISION" in results[2] and "FAIL" in results[2]
assert "NO_MATCHING_FLIGHT" in results[3] and "FL101" in results[3] and "NEEDS REVISION" in results[3]
assert results[4].startswith("No data available for this destination")
print("All five workflow checks passed.")

All five workflow checks passed.


## Run your own message

Replace the string and run the cell. The reply is the Constraint Checker output.


In [ ]:
my_trip = plan_trip(
    "Plan a trip from SFO to Lisbon on Oct 12, 2026. 3 nights, total budget $900. "
    "I need a direct flight, wheelchair assistance, and a hotel with a wheelchair-accessible room and step-free entrance."
)

=== Coordinator ===
DESTINATION: LIS
TRAVEL_DATE: 2026-10-12
BUDGET_TOTAL_USD: 900
DIRECT_FLIGHT_REQUIRED: yes
ACCESSIBILITY_NEEDS: wheelchair assistance, wheelchair-accessible room, step-free entrance
NIGHTS: 3
NOTES: none

=== Flight Agent ===
RECOMMENDED_FLIGHT_ID: FL101
AIRLINE: SkyWay Air
DEPARTURE: 2026-10-12 08:15
PRICE_USD: 410
STOPS: 0
ACCESSIBILITY_MATCH: yes, wheelchair_assistance is yes
ALTERNATE_OPTION: FL103

=== Itinerary Agent ===
SUMMARY: LIS on 2026-10-12, 3 nights
FLIGHT: FL101, SkyWay Air, $410, 0 stops
HOTEL: HT201, 0, $140/night x 3 = $420
TOTAL_ESTIMATED_COST_USD: 830
TRAVELER_CONSTRAINTS: budget $900, direct flight yes, accessibility wheelchair assistance, wheelchair-accessible room, step-free entrance
OPEN_ITEMS: none

=== Constraint Checker ===
ITINERARY_UNDER_REVIEW:
FLIGHT: FL101, SkyWay Air, $410, 0 stops
HOTEL: HT201, 0, $140/night x 3 = $420
TOTAL: $830
BUDGET_CHECK: PASS, $410 + $420 = $830 vs budget $900
DIRECT_FLIGHT_CHECK: PASS, 0 stops
ACCESSIBILITY_